In [ ]:
# Parameters (toggle cell as parameter cell in Fabric UI if driving from a pipeline)
source_path = "Files/landing/titanic/titanic.csv"
bronze_table = "bronze_titanic"
pipeline_run_id = "manual_run"   # overridden by pipeline @pipeline().RunId

In [ ]:
from pyspark.sql.functions import current_timestamp, input_file_name, lit
from pyspark.sql.types import StringType, IntegerType, DoubleType, StructType, StructField

## Bronze Layer — Titanic

**Purpose:** Land the raw CSV as-is (schema-on-read) into a Delta table, with audit columns.

**Rules:**
- No business logic. No type fixes. No renames.
- Every row gets ingestion metadata for lineage.
- Overwrite mode is fine for a demo; in production use append + a watermark.

In [ ]:
# Explicit schema = predictable Bronze. Strings for everything "dirty"; fix types in Silver.
raw_schema = StructType([
    StructField("PassengerId", IntegerType(), True),
    StructField("Survived",    IntegerType(), True),
    StructField("Pclass",      IntegerType(), True),
    StructField("Name",        StringType(),  True),
    StructField("Sex",         StringType(),  True),
    StructField("Age",         DoubleType(),  True),   # has nulls
    StructField("SibSp",       IntegerType(), True),
    StructField("Parch",       IntegerType(), True),
    StructField("Ticket",      StringType(),  True),
    StructField("Fare",        DoubleType(),  True),
    StructField("Cabin",       StringType(),  True),
    StructField("Embarked",    StringType(),  True),
])

df_raw = (
    spark.read
         .option("header", "true")
         .option("quote", '"')
         .option("escape", '"')
         .schema(raw_schema)
         .csv(source_path)
)

print(f"Rows read from source: {df_raw.count()}")
df_raw.printSchema()

In [ ]:
# Add audit columns
df_bronze = (
    df_raw
    .withColumn("_ingested_at",   current_timestamp())
    .withColumn("_source_file",   input_file_name())
    .withColumn("_pipeline_run_id", lit(pipeline_run_id))
)

In [ ]:
# Write as Delta. Overwrite keeps the demo idempotent.
(
    df_bronze.write
             .format("delta")
             .mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable(bronze_table)
)

print(f"Bronze table '{bronze_table}' written.")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {bronze_table}").show()
spark.sql(f"SELECT * FROM {bronze_table} LIMIT 5").show(truncate=False)